### Colab Activity 10:2: Forecasting with Decomposition Models

You have now seen how to both decompose a time series into seasonal and trend components, and how they can be used to forecast into the future using statsmodels.  In this activity, your goal is to identify a new (to you) time series dataset and build a forecast using a seasonal and trend additive or multiplicative model using statsmodels.  You will summarize your findings in an executive brief that explores the following:

- **Data Description**: A high-level overview of your data, its timeframe, and general information on your dataset.
- **Visualizations**: An overview of your findings and conclusions through different types of plots. 
- **Forecast**: A description of the forecast.  Describe the period that was projected and what the forecast says about your data.  Be sure to include presentation-ready plots with appropriate labels and titles.
- **Uncertainty**: Discuss the evaluation of your model on testing data and explore the residuals.  Discuss the consequence of this error for your model and forecasts.  Is there still structure to uncover?  




Suggested resources for data:

1. [Bureau of Labor Statistics](https://www.bls.gov/): Contains numerous time series such as Consumer Price Index and Inflation indicies.
2. [Kaggle](https://www.kaggle.com/search?q=time+series): Contains numerous time series datasets from different contexts.
3. [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets.php?format=&task=&att=&area=&numAtt=&numIns=&type=ts&sort=nameUp&view=table): Many time series example datasets.


Data was pulled from https://www.kaggle.com/datasets/bobnau/daily-website-visitors

This data set contains information concerning user visits and page loads on a website. The data differentiates from first-time visitors, unique visits and returning visits. My analysis will be focused on forecasting page loads. I will set aside 2020 as our test set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import acf

In [ ]:
visitors = pd.read_csv("module 10/colab_activity10_2_starter/data/visitors.csv")
visitors.describe()

In [ ]:
visitors.info()

In [ ]:
visitors.sample(5)

In [ ]:
visitors.isna().mean()

In [ ]:
visitors["date"] = pd.to_datetime(visitors["Date"])
visitors["Page.Loads"] = visitors["Page.Loads"].str.replace(",", "")
visitors["loads"] = visitors["Page.Loads"].astype("int")
visitors = visitors.drop(
    [
        "Row",
        "Day",
        "Date",
        "Day.Of.Week",
        "Page.Loads",
        "Unique.Visits",
        "First.Time.Visits",
        "First.Time.Visits",
        "Returning.Visits",
    ],
    axis=1,
)
visitors = visitors.set_index("date")

In [ ]:
visitors.tail()

In [ ]:
fig = px.line(visitors.reset_index(), x="date", y="loads", title="Page Loads")
fig.show()
fig.write_image("module 10/colab_activity10_2_starter/images/page_loads.png")

In [ ]:
# Let's save 2020 as test set
y_train = visitors[:"2019"]
y_test = visitors["2020":]

print(y_train.head())
print(y_train.tail())
print(y_test.head())
print(y_test.tail())

In [ ]:
plt.plot(y_train, label="historical")
plt.plot(y_test, label="future")
plt.grid()
plt.legend()
plt.savefig("module 10/colab_activity10_2_starter/images/page_loads_split.png")

In [ ]:
# Extract trend
stl = STL(y_train, period=12)
results = stl.fit()

## Answer check
plt.plot(results.trend)
plt.grid()
plt.title("Trend of page loads")
plt.savefig("module 10/colab_activity10_2_starter/images/page_loads_trend.png")

In [ ]:
# plot residuals
plt.plot(results.resid)
plt.grid()
plt.title("Residuals");

In [ ]:
plot_acf(acf(results.resid, nlags=100))

In [ ]:
res = seasonal_decompose(y_train, model="additive", period=30)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 8))
res.trend.plot(ax=ax1, ylabel="trend")
res.resid.plot(ax=ax2, ylabel="seasonality")
res.seasonal.plot(ax=ax3, ylabel="residual")
plt.show()

In [ ]:
plot_acf(res.seasonal)

In [ ]:
# instantiate
stlf = STLForecast(
    y_train, ARIMA, model_kwargs={"seasonal_order": (0, 0, 0, 0), "freq": "D"}
)
# fit model using historical data
stlf_results = stlf.fit()
# produce forecast for future data
forecast = stlf_results.forecast(len(y_test))

In [ ]:
plt.plot(y_test, label="true future data")
plt.plot(forecast, label="forecast")
plt.plot(y_train, label="training data")
plt.legend()
plt.title("Forecast with STL and Future Data")
plt.grid();

In [ ]:
pred_error = y_test.loads - forecast
mae = np.abs(pred_error).mean()
rmse = np.sqrt(np.square(pred_error).mean())

# Answer check
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")